<a href="https://colab.research.google.com/github/AcSsalazar/the-color-of-emotions/blob/main/Notebooks-EN/4.2-fine-tuning-png-vs-tensor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fine-Tuning with ResNet-18 and EfficientNet-B0

### Introduction: PNG vs. Tensor

This is the stage before selecting the best model for **Speech Emotion Recognition (SER)**. In this notebook, we train models using two different input approaches:

1. **Images (PNG):** Mel spectrograms previously generated and saved as image files.
2. **3D Tensors:** Data processed directly with PyTorch from **Cepstral Coefficients (MFCCs)** and their **first- and second-order deltas**. This produces an input tensor with shape `[3, Frequency, Time]`.

We analyze performance metrics, computational costs, and above all the reach of models trained on compressed images compared with those trained on tensors, which preserve the information and temporal dimension in their inputs more faithfully.

In [ ]:
# Imports
#----------------------------------------------------------------
import os
import gc
import numpy as np
import glob
import torch
import random
import torch.nn as nn
import seaborn as sns
import torch.optim as optim
import matplotlib.pyplot as plt
#----------------------------------------------------------------
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms, models
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from torch.cuda.amp import GradScaler, autocast
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
# Hardware optimizations and seed fixing for reproducibility
torch.manual_seed(42)
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device en uso: {device}")

In [ ]:
# Copy the entire features folder from Drive to Colab's ultra-fast disk
#!cp -r /content/drive/MyDrive/ravdess_images_02/ /content/features_local
os.makedirs('/content/ravdess_and_crema_images', exist_ok=True)
#os.makedirs('/content/data_split_tensors', exist_ok=True)
# Optional: If you have a .zip file in Drive, it is EVEN FASTER to copy the .zip and unzip it locally:
!cp /content/drive/MyDrive/mel_spec_and_mfcc_images.zip /content/mel_spec_mfcc_images.zip
!cp -r /content/drive/MyDrive/data_split_tensors/ /content

!unzip -q /content/mel_spec_mfcc_images.zip -d /content/ravdess_and_crema_images

In [ ]:
# Configuraciones y rutas
BASE_DIR_IMG = '/content/ravdess_and_crema_images'
BASE_DIR_TENSOR = '/content/data_split_tensors'
MODELS_SAVE_DIR = '/content/saved_models'
os.makedirs(MODELS_SAVE_DIR, exist_ok=True)

BATCH_SIZE = 64 # Ajustable si hay problemas de memoria con DenseNet (Spoiler: Dejar en 64)

# It is recommended to use the GPU provided by Google Colab to speed up training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Print root_dir content:
img_content = (os.listdir(BASE_DIR_IMG))
tensor_content = (os.listdir(BASE_DIR_TENSOR))

print(f"Contenido de {BASE_DIR_IMG}: {img_content}")
print(f"Contenido de {BASE_DIR_TENSOR}: {tensor_content}")

### Dataloaders from split data (Train, Validation, Test)

In [ ]:
def get_images_dataloaders(feature_type):


  """
  Construye DataLoaders aislando Train/Val/Test.
  Apply ImageNet normalization strictly required for transfer learning.
  Note: ImageFolder automatically converts RGBA to RGB (black background) by default.
  """
  feature_dir_img = os.path.join(BASE_DIR_IMG, feature_type)

  transform_pipeline = transforms.Compose([
      #transforms.Resize(224,224), # Change if the source image is not 224x224
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
  ])

  datasets_dict = {}
  for split in ['train', 'val', 'test']:
      split_path = os.path.join(feature_dir_img, split)
      datasets_dict[split] = datasets.ImageFolder(root=split_path, transform=transform_pipeline)


  # Optimizations: num_workers=2 and pin_memory=True speed up GPU transfer
  train_loader = DataLoader(datasets_dict['train'], batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
  val_loader = DataLoader(datasets_dict['val'], batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
  test_loader = DataLoader(datasets_dict['test'], batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

  class_names = datasets_dict['train'].classes
  print(f"[{feature_type.upper()}] Clases detectadas: {class_names}")
  print(f"Muestras -> Train: {len(datasets_dict['train'])} | Val: {len(datasets_dict['val'])} | Test: {len(datasets_dict['test'])}")

  return train_loader, val_loader, test_loader, class_names

In [ ]:
get_images_dataloaders("mfcc")

In [ ]:
# Map indices to classes just like in extraction
IDX_TO_CLASS = {
    0: 'angry',
    1: 'disgust',
    2: 'fearful',
    3: 'happy',
    4: 'neutral',
    5: 'sad',
    6: 'surprised'
}

CLASS_NAMES = list(IDX_TO_CLASS.values())

def get_tensors_dataloaders(BASE_DIR_TENSOR, BATCH_SIZE):
    datasets_dict = {}

    file_map = {
        'train': 'train_tensors.pt',
        'val': 'val_tensors.pt',
        'test': 'test_tensors.pt'
    }

    for split, filename in file_map.items():
        split_path = os.path.join(BASE_DIR_TENSOR, filename)
        if not os.path.exists(split_path):
            raise FileNotFoundError(f"File does not exist de split: {split_path}")
        datasets_dict[split] = torch.load(split_path)

    # If you use GPU, pin_memory helps; on CPU it does not
    pin = torch.cuda.is_available()

    train_loader = DataLoader(
        datasets_dict['train'],
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=pin
    )
    val_loader = DataLoader(
        datasets_dict['val'],
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=pin
    )
    test_loader = DataLoader(
        datasets_dict['test'],
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=pin
    )

    class_names = [IDX_TO_CLASS[i] for i in sorted(IDX_TO_CLASS.keys())]

    print(f"Clases detectadas: {class_names}")
    print(
        f"Muestras -> Train: {len(datasets_dict['train'])} | "
        f"Val: {len(datasets_dict['val'])} | "
        f"Test: {len(datasets_dict['test'])}"
    )

    # sanity check of the first tensor shape
    x0, y0 = datasets_dict['train'][0]
    print(f"Shape ejemplo train[0]: {tuple(x0.shape)} | label={y0}")

    return train_loader, val_loader, test_loader, class_names


In [ ]:
get_tensors_dataloaders(BASE_DIR_TENSOR, BATCH_SIZE)

### Model Factory for Tensors

ResNet18 y Efficientnet-b0

In [ ]:
class TensorModelFactory:
  @staticmethod
  def get_model(model_name: str, num_classes: int, freeze_base: bool=True):
    if model_name == 'resnet18':
      model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
      # Adaptation for small input [3,13,94]
      model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
      model.maxpool = nn.Identity()
      if freeze_base:
        for param in model.parameters():
          param.requires_grad = False

      in_features = model.fc.in_features
      model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_features, num_classes))

    elif model_name == 'efficientnet_b0':
      model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)


      if freeze_base:
          for p in model.parameters():
              p.requires_grad = False

      in_f = model.classifier[1].in_features
      model.classifier = nn.Sequential(
          nn.Dropout(0.4),
          nn.Linear(in_f, num_classes)
      )

    else:
        raise ValueError(f"Model not supported for tensor pipeline: {model_name}")

    return model.to(device)

### Neural network training

Function to train with ResNet or EfficientNet-B0. The model training functions are very similar; in them we can configure network parameters, for example:
* Directory where we save the models `save_path`,
* Number of epochs `epochs`,
* Learning rate `lr`,
* Number of attempts before stopping training `patience`,
* Weight configuration `weight_decay`.

In [ ]:

def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def cleanup_state():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def train_model_tensor(
    model,
    train_loader,
    val_loader,
    save_path,
    epochs=20,
    lr=1e-3,
    patience=4,
    weight_decay=1e-2
):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=lr, weight_decay=weight_decay)

    # scheduler maximiza F1 macro
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2
    )
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

    # best checkpoint por F1 macro
    best_val_f1 = -1.0
    trigger = 0

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            train_correct += (preds == y).sum().item()
            train_total += y.size(0)

        train_loss /= max(train_total, 1)
        train_acc = train_correct / max(train_total, 1)

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        all_val_y, all_val_p = [], []

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                    logits = model(x)
                    loss = criterion(logits, y)

                val_loss += loss.item() * x.size(0)
                preds = logits.argmax(dim=1)

                val_correct += (preds == y).sum().item()
                val_total += y.size(0)
                all_val_y.extend(y.cpu().numpy())
                all_val_p.extend(preds.cpu().numpy())

        val_loss /= max(val_total, 1)
        val_acc = val_correct / max(val_total, 1)
        val_f1_macro = f1_score(all_val_y, all_val_p, average='macro')

        scheduler.step(val_f1_macro)
        lr_now = optimizer.param_groups[0]['lr']

        print(f"[{epoch+1:02d}/{epochs}] lr={lr_now:.1e} | "
              f"train_loss={train_loss:.4f} acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} acc={val_acc:.4f} f1m={val_f1_macro:.4f}")

        if val_f1_macro > best_val_f1:
            best_val_f1 = val_f1_macro
            torch.save(model.state_dict(), save_path)
            trigger = 0
        else:
            trigger += 1
            print(f"Early stop activado: {trigger}/{patience}")
            if trigger >= patience:
                print("Early stopping.")
                break

    return save_path

Training run

In [ ]:
# State cleanup to avoid memorized results
set_global_seed(42)
cleanup_state()

In [ ]:
MODELS_SAVE_DIR = "/content/saved_models"
os.makedirs(MODELS_SAVE_DIR, exist_ok=True)
MODEL_ARCH_1 = 'efficientnet_b0'
exp_name_b0 = f"tensor{MODEL_ARCH_1}"

train_loader, val_loader, test_loader, class_names = get_tensors_dataloaders(
    BASE_DIR_TENSOR,
    BATCH_SIZE
)

phase1_path = os.path.join(MODELS_SAVE_DIR, f"{exp_name_b0}efficient_fase1.pth")

model = TensorModelFactory.get_model(MODEL_ARCH_1, num_classes=len(CLASS_NAMES), freeze_base=True)

best_phase1_eff = train_model_tensor(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    save_path=phase1_path,
    epochs=15,
    lr=1e-3,
    patience=4
)

print("Phase 1 best model:", best_phase1_eff)

In [ ]:
# Descongelamiento de las capas de la CNN
phase2_path = os.path.join(MODELS_SAVE_DIR, f"{exp_name_b0}_fase2.pth")

model.load_state_dict(torch.load(best_phase1_eff, map_location=device, weights_only=True))
for p in model.parameters():
    p.requires_grad = True

best_phase2_eff = train_model_tensor(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    save_path=phase2_path,
    epochs=25,
    lr=1e-5,
    patience=4
)

print("Phase 2 best efficient model:", best_phase2_eff)

Model evaluation

In [ ]:

def evaluate_tensor_model_eff(model_arch, model_path, test_loader, class_names):
    model = TensorModelFactory.get_model(model_arch, num_classes=len(class_names), freeze_base=False)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()

    all_y, all_p = [], []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                logits = model(x)
            preds = logits.argmax(dim=1)

            all_y.extend(y.cpu().numpy())
            all_p.extend(preds.cpu().numpy())

    acc = accuracy_score(all_y, all_p)
    f1m = f1_score(all_y, all_p, average='macro')
    print(f"Test Accuracy: {acc:.4f} | Test F1 macro: {f1m:.4f}\n")
    print(classification_report(all_y, all_p, target_names=class_names))

    cm = confusion_matrix(all_y, all_p)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - {model_arch} (tensor)")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    del model
    torch.cuda.empty_cache()
    gc.collect()

evaluate_tensor_model_eff(MODEL_ARCH_1, best_phase2_eff, test_loader, CLASS_NAMES)

### Fine-tuning with ResNet18

In [ ]:
set_global_seed(42)
cleanup_state()

In [ ]:
MODELS_SAVE_DIR = "/content/saved_models"
os.makedirs(MODELS_SAVE_DIR, exist_ok=True)
MODEL_ARCH_2 = 'resnet18'
exp_name_18 = f"tensor_{MODEL_ARCH_2}"

train_loader, val_loader, test_loader, class_names = get_tensors_dataloaders(
    BASE_DIR_TENSOR,
    BATCH_SIZE
)



phase1_path = os.path.join(MODELS_SAVE_DIR, f"{exp_name_18}resnet_fase1.pth")

model = TensorModelFactory.get_model(MODEL_ARCH_2, num_classes=len(CLASS_NAMES), freeze_base=True)

best_fase1_18 = train_model_tensor(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    save_path=phase1_path,
    epochs=15,
    lr=1e-3,
    patience=4
)

print("Phase 1 best model:", best_fase1_18)

In [ ]:
phase2_path = os.path.join(MODELS_SAVE_DIR, f"{exp_name_18}_phase2_unfrozen.pth")

model.load_state_dict(torch.load(best_fase1_18, map_location=device, weights_only=True))
for p in model.parameters():
    p.requires_grad = True

best_fase2_18 = train_model_tensor(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    save_path=phase2_path,
    epochs=25,
    lr=1e-5,
    patience=4
)

print("Phase 2 best ResNet model:", best_fase2_18)

In [ ]:
def evaluate_tensor_model(model_arch, model_path, test_loader, class_names):
    model = TensorModelFactory.get_model(model_arch, num_classes=len(class_names), freeze_base=False)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()

    all_y, all_p = [], []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                logits = model(x)
            preds = logits.argmax(dim=1)

            all_y.extend(y.cpu().numpy())
            all_p.extend(preds.cpu().numpy())

    acc = accuracy_score(all_y, all_p)
    f1m = f1_score(all_y, all_p, average='macro')
    print(f"Test Accuracy: {acc:.4f} | Test F1 macro: {f1m:.4f}\n")
    print(classification_report(all_y, all_p, target_names=class_names))

    cm = confusion_matrix(all_y, all_p)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - {model_arch} (tensor)")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    del model
    torch.cuda.empty_cache()
    gc.collect()

evaluate_tensor_model(MODEL_ARCH_2, best_fase2_18, test_loader, CLASS_NAMES)

### Model Factory for Images

ResNet18 y Efficientnet-b0

In [ ]:
set_global_seed(42)
cleanup_state()

In [ ]:
class ImageModelFactory:
    @staticmethod
    def get_model(model_name, num_classes, freeze_base=True):
        """
        Dynamically instantiate convolutional architectures and adapt the final layer.
        """
        if model_name == 'resnet18':
            model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            if freeze_base:
                for param in model.parameters(): param.requires_grad = False

            num_ftrs = model.fc.in_features
            model.fc = nn.Sequential(nn.Dropout(0.7), nn.Linear(num_ftrs, num_classes))

        elif model_name == 'efficientnet_b0':
            model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            if freeze_base:
                for param in model.parameters(): param.requires_grad = False

            num_ftrs = model.classifier[1].in_features
            model.classifier[1] = nn.Sequential(nn.Dropout(0.5), nn.Linear(num_ftrs, num_classes))

        else:
            raise ValueError(f"Arquitectura {model_name} no soportada.")

        return model.to(device)

In [ ]:
def train_model(model, train_loader, val_loader, model_save_name, epochs=15, lr=1e-3, patience=5):
    """
    Training loop with Mixed Precision (AMP), Early Stopping, and LR Scheduler.
    """
    criterion = nn.CrossEntropyLoss()
    # Optimize only the parameters that require gradients (useful in Phase 1)
    # Increase weight_decay to 1e-2 to combat observed overfitting
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=5e-2)

    # Configure dynamic LR: reduce LR by half if val_loss does not improve in 2 epochs
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    scaler = torch.amp.GradScaler('cuda')

    best_val_loss = float('inf')
    trigger_times = 0
    save_path = os.path.join(MODELS_SAVE_DIR, f'{model_save_name}.pth')

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            # Forward pass with AMP for memory savings
            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / total
        train_acc = correct / total

        # Validation phase
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss = val_loss / val_total
        val_acc = val_correct / val_total

        # Update the scheduler based on the validation metric
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch {epoch+1}/{epochs}[LR:{current_lr:.1e}] | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        # Early stopping logic
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)

        else:
            trigger_times += 1
            print(f"Early stop trigger activado {trigger_times} / {patience}")
            if trigger_times >= patience:
                print(f"--> Early stopping triggered at epoch {epoch+1}.")
                break

    return save_path

In [ ]:

MODES = ['vector', 'png']
TARGET_FEATURE = 'mel_spec' # Opciones: 'mel_spec', 'mfcc'
MODEL_ARCHITECTURE = 'efficientnet_b0' # Opciones: 'resnet18', 'efficientnet_b0'

# Ensure 'device' refers to the torch.device object, not the sklearn function

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Data loading (use the optimized DataLoaders)
print(f"Loading data from: {TARGET_FEATURE.upper()}")
train_loader, val_loader, test_loader, class_names = get_images_dataloaders(TARGET_FEATURE)

# 2. Dynamic construction via the Model Factory
model = ImageModelFactory.get_model(model_name=MODEL_ARCHITECTURE, num_classes=len(class_names), freeze_base=True)

experiment_name = f"{TARGET_FEATURE}_{MODEL_ARCHITECTURE}"

# =====================================================================
# PHASE 1: TRANSFER LEARNING (Top classifier)
# =====================================================================
print(f"\n--- INICIANDO FASE 1: Entrenando clasificador de {MODEL_ARCHITECTURE} ---")
best_model_path_phase1 = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    model_save_name=f"{experiment_name}_phase1",
    epochs=15,
    lr=1e-3, # Higher LR because we only train the final linear layer
    patience=4
)

# =====================================================================
# PHASE 2: DEEP FINE-TUNING (Unfreezing)
# =====================================================================
print(f"\n--- INICIANDO FASE 2: Fine-Tuning profundo ---")
# Restore the model to the lowest-loss point in Phase 1
model.load_state_dict(torch.load(best_model_path_phase1, weights_only=True))

# Unfreeze ALL weights so convolutional filters adapt to the audio
for param in model.parameters():
    param.requires_grad = True

best_model_path_phase2 = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    model_save_name=f"{experiment_name}_phase2_unfrozen",
    epochs=40,
    lr=1e-5, # CRITICAL: much lower LR to avoid destroying ImageNet weights
    patience=4
)

print(f"\nTraining finished. Best model saved at: {best_model_path_phase2}")

In [ ]:
def evaluate_model(model_path, test_loader, class_names, model_arch):
    """
    Load weights of a trained model and run inference on the test set
    and generates standardized classification metrics.
    """
    print(f"Loading model for evaluation: {model_path}")

    # Instantiate an empty architecture (no freezing) and load the final weights
    model = ImageModelFactory.get_model(model_name=model_arch, num_classes=len(class_names), freeze_base=False)
    model.load_state_dict(torch.load(model_path, weights_only=True))
    model.to(device)
    model.eval() # Evaluation mode: freezes Dropout and BatchNorm layers

    all_preds = []
    all_labels = []

    # Disabling gradient tracking is required to free VRAM during inference
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # AMP is also used in inference for greater speed
            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)

            # Data handling: Scikit-learn requires arrays on CPU and in NumPy
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 1. Statistical report (F1-score, precision, recall)
    print("\n" + "="*60)
    print(f"CLASSIFICATION REPORT: {model_arch.upper()}")
    print("="*60)
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # 2. Visualization: Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {model_arch.upper()}')
    plt.ylabel('Etiqueta Real')
    plt.xlabel('CNN prediction')
    plt.tight_layout()
    plt.show()

    # Memory management: GPU cleanup after inference
    del model
    torch.cuda.empty_cache()
    gc.collect()

# Run evaluation with the newly trained model
evaluate_model(
    model_path=best_model_path_phase2,
    test_loader=test_loader,
    class_names=class_names,
    model_arch=MODEL_ARCHITECTURE
)

Resnet18 con melspec images

In [ ]:
set_global_seed(42)
cleanup_state()

In [ ]:
MODES = ['vector', 'png']
TARGET_FEATURE = 'mel_spec' # Opciones: 'mel_spec', 'mfcc'
MODEL_ARCHITECTURE = 'resnet18' # Opciones: 'resnet18', 'efficientnet_b0'

# Ensure 'device' refers to the torch.device object, not the sklearn function

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Data loading (use the optimized DataLoaders)
print(f"Loading data from: {TARGET_FEATURE.upper()}")
train_loader, val_loader, test_loader, class_names = get_images_dataloaders(TARGET_FEATURE)

# 2. Dynamic construction via the Model Factory
model = ImageModelFactory.get_model(model_name=MODEL_ARCHITECTURE, num_classes=len(class_names), freeze_base=True)

experiment_name = f"{TARGET_FEATURE}_{MODEL_ARCHITECTURE}"

# =====================================================================
# PHASE 1: TRANSFER LEARNING (Top classifier)
# =====================================================================
print(f"\n--- INICIANDO FASE 1: Entrenando clasificador de {MODEL_ARCHITECTURE} ---")
best_model_path_phase1 = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    model_save_name=f"{experiment_name}_phase1",
    epochs=15,
    lr=1e-3, # Higher LR because we only train the final linear layer
    patience=4
)

# =====================================================================
# PHASE 2: DEEP FINE-TUNING (Unfreezing)
# =====================================================================
print(f"\n--- INICIANDO FASE 2: Fine-Tuning profundo ---")
# Restore the model to the lowest-loss point in Phase 1
model.load_state_dict(torch.load(best_model_path_phase1, weights_only=True))

# Unfreeze ALL weights so convolutional filters adapt to the audio
for param in model.parameters():
    param.requires_grad = True

best_model_path_phase2 = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    model_save_name=f"{experiment_name}_phase2_unfrozen",
    epochs=40,
    lr=1e-5, # CRITICAL: much lower LR to avoid destroying ImageNet weights
    patience=4
)

print(f"\nTraining finished. Best model saved at: {best_model_path_phase2}")

In [ ]:
def evaluate_model(model_path, test_loader, class_names, model_arch):
    """
    Load weights of a trained model and run inference on the test set
    and generates standardized classification metrics.
    """
    print(f"Loading model for evaluation: {model_path}")

    # Instantiate an empty architecture (no freezing) and load the final weights
    model = ImageModelFactory.get_model(model_name=model_arch, num_classes=len(class_names), freeze_base=False)
    model.load_state_dict(torch.load(model_path, weights_only=True))
    model.to(device)
    model.eval() # Evaluation mode: freezes Dropout and BatchNorm layers

    all_preds = []
    all_labels = []

    # Disabling gradient tracking is required to free VRAM during inference
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # AMP is also used in inference for greater speed
            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)

            # Data handling: Scikit-learn requires arrays on CPU and in NumPy
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 1. Statistical report (F1-score, precision, recall)
    print("\n" + "="*60)
    print(f"CLASSIFICATION REPORT: {model_arch.upper()}")
    print("="*60)
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # 2. Visualization: Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {model_arch.upper()}')
    plt.ylabel('Etiqueta Real')
    plt.xlabel('CNN prediction')
    plt.tight_layout()
    plt.show()

    # Memory management: GPU cleanup after inference
    del model
    torch.cuda.empty_cache()
    gc.collect()

# Run evaluation with the newly trained model
evaluate_model(
    model_path=best_model_path_phase2,
    test_loader=test_loader,
    class_names=class_names,
    model_arch=MODEL_ARCHITECTURE
)

In [ ]:
import shutil
# Save the models for the next stage (late fusion)
output_filename = '/content/saved_models'
shutil.make_archive(output_filename, 'zip', MODELS_SAVE_DIR)

# Rename the zip file so it has the .zip extension
# This is a workaround for make_archive, which does not add the extension by default if base_name already looks like one.
import os
os.rename('/content/saved_models.zip', '/content/saved_models_final.zip')

!cp /content/saved_models_final.zip /content/drive/MyDrive/saved_models_img_and_tensor.zip